In [1]:
import numpy as np
import pandas as pd
TARGETS = ["X"]

SETS = [
    "Train_1",
    "Train_2",
    "Val",
    "Test_1",
    "Test_2",
    "Test_3"
]

results = pd.read_excel("resultados.xlsx")


In [2]:
for dataset in SETS:
    for target in TARGETS:
        col_r2 = f"R2_{dataset}_{target}"
        col_mse = f"MSE_{dataset}_{target}"
        
        for col in [col_r2, col_mse]:
            if col in results.columns:
                results[col] = (
                    results[col]
                    .astype(str)
                    .str.strip("[]")
                    .astype(float)
                )

In [7]:
best_models_tables = {}
best_only = []

summary_all = []     # estatísticas com todos
summary_topN = []   # estatísticas com top 10

N = 10

import pandas as pd

def summarize_models(results, TARGETS, SETS, N=10, output="Resumo_Estatisticas.xlsx"):
    
    best_models_tables = {}
    best_only = []
    summary_all = []
    summary_top10 = []

    for target in TARGETS:
        for dataset in SETS:

            col_r2 = f"R2_{dataset}_{target}"
            col_mse = f"MSE_{dataset}_{target}"

            if col_r2 not in results.columns:
                continue

            table = results.copy()

            table = table.sort_values(by=col_r2, ascending=False)

            best_models_tables[f"{dataset}_{target}"] = table

            # 🔹 melhor modelo
            best = table.iloc[0]

            best_only.append({
                "Target": target,
                "Dataset": dataset,
                "Best_Model": best["model"],
                "Neurons": best["Neurons"],
                "r": best["r"],
                "Best_R2": best[col_r2]
            })

            # 🔹 estatísticas de todos
            summary_all.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": table[col_r2].mean(),
                "std_r2": table[col_r2].std(),
                "mean_mse": table[col_mse].mean(),
                "std_mse": table[col_mse].std()
            })

            # 🔹 estatísticas top N
            topN = table.head(N)

            summary_topN.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": topN[col_r2].mean(),
                "std_r2": topN[col_r2].std(),
                "mean_mse": topN[col_mse].mean(),
                "std_mse": topN[col_mse].std()
            })

    df_summary_all = pd.DataFrame(summary_all)
    df_summary_topN = pd.DataFrame(summary_topN)
    best_only_df = pd.DataFrame(best_only)
    # exportar
    with pd.ExcelWriter(output) as writer:
        df_summary_all.to_excel(writer, sheet_name="Todos_Modelos", index=False)
        df_summary_topN.to_excel(writer, sheet_name="Top_Modelos", index=False)
        best_only_df.to_excel(writer, sheet_name="Melhor_Modelo", index=False)

    return best_models_tables, df_summary_all, df_summary_topN, best_only_df

In [8]:
tables, summary_all, summary_top10, best_models = summarize_models(
    results, TARGETS, SETS
)

In [9]:
summary_top10["bestR2"] = best_models["Best_R2"]
summary_top10["Neurons"] = best_models["Neurons"]
summary_top10["r"] = best_models["r"]

display(summary_top10)

,dataset,target,mean_r2,std_r2,mean_mse,std_mse,bestR2,Neurons,r
0,Train_1,X,0.965426,0.006905,0.003728,0.000745,0.980419,"[8, 4]",0.01
1,Train_2,X,0.981990,0.006788,0.001518,0.000572,0.990827,"[16, 8]",0.01
2,Val,X,0.881102,0.017154,0.011369,0.001640,0.900396,"[32, 16]",0.01
3,Test_1,X,0.971977,0.005967,0.002936,0.000625,0.984707,"[64, 32]",0.01
4,Test_2,X,0.987633,0.003724,0.001018,0.000306,0.992973,"[128, 64]",0.01
5,Test_3,X,0.173114,0.228863,0.076077,0.021056,0.641939,[128],0.90


- Entradas: SWdRef, SWeRef, Theta
- Saidas: X
- Trajetorias: ZZ + ZZRoted